# Session 11 — Your turn: Attention

Attention is three matrix products and a softmax. You will write it for a three-word example, then add the causal mask a text generator needs.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np

tokens = ["robot", "battery", "it"]
Q = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 2.0]])      # what each token asks for
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.2]])      # what each token offers to be found by
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])      # what each token passes on when it is chosen
from aij_check import check, close, summary

## Task 1: Attention by hand

Write `softmax(scores)` (over the last axis, subtracting the row maximum first for stability) and compute `scores = Q @ K.T / sqrt(d)` with `d` the number of columns of `K`,
`weights = softmax(scores)`, and `output = weights @ V`. Then set `it_looks_at` to the token that "it" (the last row) gives the largest weight.

In [ ]:
def softmax(s):
    ...


d = ...
scores = ...
weights = ...
output = ...
it_looks_at = ...
print(np.round(weights, 2), it_looks_at)

In [ ]:
check("every row of weights sums to 1", lambda: close(weights.sum(axis=1), 1.0, 1e-9))
check("the weights match the reference", lambda: close(weights, __import__("scipy.special", fromlist=["softmax"]).softmax(Q @ K.T / np.sqrt(2), axis=1), 1e-9),
      "scores = Q @ K.T / sqrt(d), then softmax along each row.")
check("'it' looks mostly at 'battery'", lambda: it_looks_at == "battery")
check("the output has one new vector per token", lambda: np.shape(output) == (3, 2) and close(output, weights @ V, 1e-9))
summary()

## Task 2: The causal mask

A model that writes text must not read the future. Build `mask` (True where a token may look: itself and earlier tokens), set the scores of forbidden positions to `-1e9` before the softmax, and compute `causal_weights`.
Then store in `first_token_row` the weights of the first token, and in `leaks` whether any weight above the diagonal is larger than 1e-6.

In [ ]:
mask = ...
causal_weights = ...
first_token_row = ...
leaks = ...
print(np.round(causal_weights, 2))

In [ ]:
check("the mask is lower-triangular", lambda: np.array_equal(mask, np.tril(np.ones((3, 3), dtype=bool))))
check("rows still sum to 1", lambda: close(causal_weights.sum(axis=1), 1.0, 1e-9))
check("the first token can only read itself", lambda: close(first_token_row, [1, 0, 0], 1e-9))
check("no token reads a later one", lambda: leaks is False)
summary()